# Week #4: Sorting a World of Bars

[Open this notebook in Colab](https://colab.research.google.com/github/neurosynthetic/grant-vogt/blob/main/notebooks/week04.ipynb)

> **Important:** Before you do any work, save your own copy to Google Drive.
>
> In Colab: **File → Save a copy in Drive**

In Week 3, you inferred several transformations from examples. This week, you will stay with one ARC-style task and build a complete solver one piece at a time.

Along the way, you will also learn a few useful pieces of Python syntax: `range(...)` for visiting list positions, negative indices such as `-1` for counting backward from the end of a list, and `continue` for skipping part of a loop. Each one will be explained when you first need it.

The central idea is a powerful one:

> **grid → color and bar heights → sorted heights → grid**

## Goals

By the end of this notebook, you will be able to:

- infer the rule of a new ARC-style task
- write a function that sorts a list
- find the shared color of the bars in a grid
- measure the height of each bar
- rebuild a grid from a color and a list of heights
- combine several tested helpers into one solver

## Setup

Run the next cell once. It provides grid drawings and the **Check Exercise** buttons. If the runtime restarts, run this cell again.

In [ ]:
#@title Setup utilities (run once)

import copy
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import ListedColormap

GRID_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25"
]
GRID_CMAP = ListedColormap(GRID_COLORS)

def draw_grid(ax, grid, title=""):
    rows = len(grid)
    cols = len(grid[0]) if rows else 0
    ax.imshow(grid, cmap=GRID_CMAP, vmin=0, vmax=9)
    font_size = max(6, min(12, 120 // max(rows, cols, 1)))
    colors_with_white_text = {0, 1, 2, 6, 9}
    for row_index, row in enumerate(grid):
        for column_index, color in enumerate(row):
            text_color = "white" if color in colors_with_white_text else "black"
            ax.text(column_index, row_index, str(color), ha="center", va="center",
                    color=text_color, fontsize=font_size, fontweight="bold")
    ax.set_title(title)
    ax.set_xticks([x - 0.5 for x in range(1, cols)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, rows)], minor=True)
    ax.grid(which="minor", color="white", linewidth=1)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

def show_grid(grid, title="Grid"):
    _, ax = plt.subplots(figsize=(4, 3))
    draw_grid(ax, grid, title)
    plt.show()

def show_pair(input_grid, output_grid, label=""):
    _, axes = plt.subplots(1, 2, figsize=(8, 3))
    draw_grid(axes[0], input_grid, "Input")
    draw_grid(axes[1], output_grid, "Output")
    if label:
        plt.suptitle(label)
    plt.tight_layout()
    plt.show()

def show_examples(examples):
    for number, example in enumerate(examples, start=1):
        show_pair(example["input"], example["output"], f"Training example {number}")

def make_check_button(check_fn):
    button = widgets.Button(description="Check Exercise", button_style="info")
    output = widgets.Output()
    def on_click(_):
        output.clear_output()
        with output:
            check_fn()
    button.on_click(on_click)
    display(button, output)

print("Setup complete.")

# The task

We created this ARC-style task for the internship. It is not from the official ARC dataset.

Study all three training examples before reading further. Each colored vertical bar is an object. The black cells form the background.

In [ ]:
training_examples = [
    {
        "input": [
            [0,0,0,0,0,0,0,0,0],
            [0,0,0,0,0,2,0,0,0],
            [0,0,0,0,0,2,0,0,0],
            [0,2,0,0,0,2,0,0,0],
            [0,2,0,0,0,2,0,2,0],
            [0,2,0,2,0,2,0,2,0],
        ],
        "output": [
            [0,0,0,0,0,0,0,0,0],
            [0,0,0,0,0,0,0,2,0],
            [0,0,0,0,0,0,0,2,0],
            [0,0,0,0,0,2,0,2,0],
            [0,0,0,2,0,2,0,2,0],
            [0,2,0,2,0,2,0,2,0],
        ],
    },
    {
        "input": [
            [0,0,0,8,0,0,0],
            [0,8,0,8,0,0,0],
            [0,8,0,8,0,0,0],
            [0,8,0,8,0,0,0],
            [0,8,0,8,0,8,0],
        ],
        "output": [
            [0,0,0,0,0,8,0],
            [0,0,0,8,0,8,0],
            [0,0,0,8,0,8,0],
            [0,0,0,8,0,8,0],
            [0,8,0,8,0,8,0],
        ],
    },
    {
        "input": [
            [0,0,0,1,0,0,0,0,0,0,0],
            [0,0,0,1,0,0,0,0,0,1,0],
            [0,0,0,1,0,0,0,1,0,1,0],
            [0,0,0,1,0,0,0,1,0,1,0],
            [0,1,0,1,0,0,0,1,0,1,0],
            [0,1,0,1,0,1,0,1,0,1,0],
            [0,1,0,1,0,1,0,1,0,1,0],
        ],
        "output": [
            [0,0,0,0,0,0,0,0,0,1,0],
            [0,0,0,0,0,0,0,1,0,1,0],
            [0,0,0,0,0,1,0,1,0,1,0],
            [0,0,0,0,0,1,0,1,0,1,0],
            [0,0,0,1,0,1,0,1,0,1,0],
            [0,1,0,1,0,1,0,1,0,1,0],
            [0,1,0,1,0,1,0,1,0,1,0],
        ],
    },
]

show_examples(training_examples)

## Observation notebook

Answer these before coding:

1. What property determines the new left-to-right order?
2. Do all bars in one grid have the same color?
3. Does that color change in the output?
4. What happens to each bar's height?
5. Do the grid dimensions change?
6. Where does the bottom of every bar remain?

### My rule

- **The bars are ordered by:**
- **The bar color:**
- **The heights:**
- **The grid dimensions:**
- **My rule in one sentence:**

## The task's format

Here are the assumptions our invented task follows:

- `0` is the black background.
- Every bar in one grid has the same nonzero color.
- Every bar is one cell wide, solid-colored, and bottom-aligned.
- A one-column black gap separates neighboring bars.
- A one-column black border appears at the left and right.
- Every bar has a different height.
- The output arranges the same bar heights from **shortest to tallest**, left to right.
- The shared bar color stays unchanged.

Stating assumptions matters. A solver can be correct for a clearly defined family of grids without solving every imaginable bar puzzle.

# Part 1: Learn to sort

Before sorting bars, practice sorting plain numbers. We will build the process one small step at a time.

## Exercise 1: Find the smallest value

Write `find_smallest(numbers)` so it returns the smallest number in a nonempty list. Start with the first number, then use a loop to compare it with every other number.

In [ ]:
def find_smallest(numbers):
    smallest = numbers[0]
    # Compare every number with smallest.
    return smallest

In [ ]:
#@title Check Exercise 1

def check_exercise_1():
    tests = [([4, 1, 5, 2], 1), ([9, 7, 8], 7), ([3], 3), ([5, -2, 0], -2)]
    try:
        for numbers, expected in tests:
            if find_smallest(numbers) != expected:
                print("Not yet. Compare each value with the smallest value seen so far.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 1 complete: you can find the smallest value!")

make_check_button(check_exercise_1)

## Exercise 2: Sort a list

Write `sort_numbers(numbers)` using this plan:

1. Copy the input with `remaining = numbers[:]`.
2. Repeat once for every input number.
3. Find the smallest value remaining.
4. Append it to `result`.
5. Remove it from `remaining` with `remaining.remove(smallest)`.

Return a new list and leave the input unchanged.

In [ ]:
def sort_numbers(numbers):
    remaining = numbers[:]
    result = []
    # Repeatedly move the smallest remaining number into result.
    return result

In [ ]:
#@title Check Exercise 2

def check_exercise_2():
    tests = [([4, 1, 5, 2], [1, 2, 4, 5]), ([3, 3, 1], [1, 3, 3]), ([7], [7]), ([], [])]
    try:
        for numbers, expected in tests:
            original = numbers[:]
            got = sort_numbers(numbers)
            if got != expected:
                print("Not yet. Repeatedly append the smallest remaining value.")
                print("Got:", got, "Expected:", expected)
                return
            if numbers != original:
                print("The values are sorted, but the input list changed. Work on a copy.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 2 complete: your sorting function works!")

make_check_button(check_exercise_2)

# Part 2: Read information from the grid

We will describe the grid with two simple pieces of information:

- one number for the shared bar color
- one list containing the bar heights

For example, a red grid might become:

```python
bar_color = 2
bar_heights = [3, 1, 5, 2]
```

## Exercise 3: Find the bar color

Write `find_bar_color(grid)` so it returns the shared nonzero color of the bars.

Scan the grid one row at a time. As soon as you find a cell that is not `0`, return that cell's value. You may assume every grid contains at least one bar.

In [ ]:
def find_bar_color(grid):
    # Scan each row and each cell.
    # Return the first color that is not 0.
    return 0

In [ ]:
#@title Check Exercise 3

def check_exercise_3():
    tests = [
        (training_examples[0]["input"], 2),
        (training_examples[1]["input"], 8),
        (training_examples[2]["input"], 1),
    ]
    try:
        for grid, expected in tests:
            got = find_bar_color(grid)
            if got != expected:
                print("Not yet. Return the first cell whose value is not 0.")
                print("Got:", got, "Expected:", expected)
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 3 complete: you found the shared bar color!")

make_check_button(check_exercise_3)

## Exercise 4: Measure one bar

Write `measure_bar(grid, column_index)` so it returns the height of the bar in the selected column.

Hints:

- Scan every row and count the cells that are not `0`.
- Because Python counts columns starting at `0`, every bar appears at an odd-numbered column index: `1`, `3`, `5`, and so on.

In [ ]:
def measure_bar(grid, column_index):
    height = 0
    # Inspect the cell in column_index of every row.
    return height

In [ ]:
#@title Check Exercise 4

def check_exercise_4():
    grid = training_examples[0]["input"]
    tests = [(1, 3), (3, 1), (5, 5), (7, 2)]
    try:
        for column_index, expected in tests:
            got = measure_bar(grid, column_index)
            if got != expected:
                print("Not yet. Count the nonzero cells in the selected column.")
                print("Got:", got, "Expected:", expected)
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 4 complete: you measured each bar's height!")

make_check_button(check_exercise_4)

# Part 3: Find every bar

Now use `measure_bar` repeatedly to build a list of all the bar heights from left to right.

## Exercise 5: Find every bar

Write `find_bars(grid)` so it returns a list of bar heights from left to right.

### Visiting every column with `range`

The expression `range(n)` produces the numbers from `0` through `n - 1`. That makes it useful for visiting every index in a list. For example:

```python
for column_index in range(len(bottom_row)):
    print(column_index)
```

This loop visits every column index from left to right.

### Skipping even columns with `continue`

In this task, bars always occupy the odd-numbered columns `1`, `3`, `5`, and so on. The even-numbered columns are always black background. You already know that `column_index % 2 == 0` is true when a column index is even.

The `continue` command skips the rest of the current loop iteration and moves directly to the next one:

```python
for column_index in range(len(bottom_row)):
    if column_index % 2 == 0:
        continue
    # Code here runs only for odd-numbered columns.
```

For each odd column, call `measure_bar` and append its result to `bar_heights`.

### A useful Python shortcut

Python lets you count backward from the end of a list using negative indices. The index `-1` means the last item, so `grid[-1]` is a clever shorthand for the grid's bottom row. Similarly, `grid[-2]` would mean the second-to-last row.

In [ ]:
def find_bars(grid):
    bar_heights = []
    bottom_row = grid[-1]

    for column_index in range(len(bottom_row)):
        if column_index % 2 == 0:
            continue
        # This point is reached only for odd-numbered columns.
        # Measure this bar and append its height to bar_heights.

    return bar_heights

In [ ]:
#@title Check Exercise 5

def check_exercise_5():
    tests = [
        (training_examples[0]["input"], [3, 1, 5, 2]),
        (training_examples[1]["input"], [4, 5, 1]),
        (training_examples[2]["input"], [3, 7, 2, 5, 6]),
    ]
    try:
        for grid, expected in tests:
            got = find_bars(grid)
            if got != expected:
                print("Not yet. Measure each odd-numbered column from left to right.")
                print("Got:     ", got)
                print("Expected:", expected)
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 5 complete: you found every bar height!")

make_check_button(check_exercise_5)

# Part 4: Turn measurements back into pixels

Now reverse the representation. Given a shared color and a list of heights, generate a new grid with a black border, black gaps, and bottom-aligned bars.

## Exercise 6: Draw the bars

Write `draw_bars(bar_heights, bar_color, grid_height)`.

Consider this plan:

1. The width is `2 * len(bar_heights) + 1`.
2. Build a black grid of the requested height and calculated width.
3. Bar number `0` belongs in column `1`, bar number `1` in column `3`, and so on. The formula is `2 * bar_index + 1`.

**Note:** Somewhat awkwardly, row #0 corresponds to the *top* of our grids, not the bottom. So, you may need to take care in how you fill these columns. There are many ways to do it!

In [ ]:
def draw_bars(bar_heights, bar_color, grid_height):
    return []

In [ ]:
#@title Check Exercise 6

def check_exercise_6():
    bar_heights = [1, 2, 3]
    expected = [[0,0,0,0,0,2,0], [0,0,0,2,0,2,0], [0,2,0,2,0,2,0]]
    try:
        got = draw_bars(bar_heights, 2, 3)
        if got != expected:
            print("Not yet. Check the grid dimensions, bar columns, and bottom alignment.")
            print("Got:     ", got)
            print("Expected:", expected)
            return
        if draw_bars([], 4, 2) != [[0], [0]]:
            print("Check what your generator does when there are no bars.")
            return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 6 complete: the color and heights became a grid!")

make_check_button(check_exercise_6)

# Part 5: Put everything together

Each helper now has one clear responsibility:

```text
                 ┌→ find_bar_color ─────────────┐
input grid ──────┤                              ├→ draw_bars → output grid
                 └→ find_bars → sort_numbers ───┘
```

A `transform_bars` function can connect them without reimplementing their logic.

## Final exercise: Write `transform_bars`

Write `transform_bars(grid)` using the four helpers. The output height is the same as the input height: `len(grid)`.

In [ ]:
def transform_bars(grid):
    # Find the color, find and sort the heights, then draw the result.
    return None

In [ ]:
#@title Check the complete transformation

def check_transform_bars():
    try:
        for number, example in enumerate(training_examples, start=1):
            original = copy.deepcopy(example["input"])
            got = transform_bars(example["input"])
            if got != example["output"]:
                print(f"Training example {number} does not match yet.")
                return
            if example["input"] != original:
                print("The answer is right, but transform_bars changed its input.")
                return

        hidden_input = [
            [0,6,0,0,0,0,0,0,0],
            [0,6,0,0,0,0,0,0,0],
            [0,6,0,0,0,6,0,0,0],
            [0,6,0,6,0,6,0,0,0],
            [0,6,0,6,0,6,0,0,0],
            [0,6,0,6,0,6,0,6,0],
            [0,6,0,6,0,6,0,6,0],
        ]
        hidden_expected = [
            [0,0,0,0,0,0,0,6,0],
            [0,0,0,0,0,0,0,6,0],
            [0,0,0,0,0,6,0,6,0],
            [0,0,0,6,0,6,0,6,0],
            [0,0,0,6,0,6,0,6,0],
            [0,6,0,6,0,6,0,6,0],
            [0,6,0,6,0,6,0,6,0],
        ]
        got = transform_bars(hidden_input)
        if got != hidden_expected:
            print("The training examples pass, but a new test grid does not.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return
    print("All checks pass. You built a complete ARC-style transformation!")

make_check_button(check_transform_bars)

## Apply the solver to a test input

This test grid was not one of the training pairs. Run the cell to see only its input. Before revealing your program's result, predict the left-to-right color order on paper.

In [ ]:
test_input = [
    [0,0,0,4,0,0,0,0,0],
    [0,4,0,4,0,0,0,0,0],
    [0,4,0,4,0,0,0,0,0],
    [0,4,0,4,0,4,0,0,0],
    [0,4,0,4,0,4,0,0,0],
    [0,4,0,4,0,4,0,4,0],
]

show_grid(test_input, "Test input")

### My prediction

- **The shared bar color:**
- **Bar heights from shortest to tallest:**

In [ ]:
test_output = transform_bars(test_input)
show_pair(test_input, test_output, "Your transformation's answer")

# Stretch challenges

Choose any that interest you:

1. **Reverse the order:** Add a parameter that can arrange bars tallest-to-shortest.
2. **Equal heights:** Decide on a clear tie-breaking rule, create examples, and update the transformation.
3. **Wider bars:** Change the parser and generator so bars may be more than one cell wide.
4. **Messier spacing:** Detect bars even when the black gaps or borders have different widths.
5. **Make a puzzle generator:** Generate valid random inputs, transform them, and display the resulting training pairs.

## Wrap-up and reflection

This week, one transformation became a complete program made of smaller, testable parts. Answer briefly:

- Why was a list of heights easier to work with than individual pixels?
- Which helper was hardest to write?
- What assumption would break your current transformation?
- Why test each helper separately before testing `transform_bars`?

### My reflection

- **The height list helped because:**
- **Hardest helper:**
- **One important assumption:**
- **Why separate tests helped:**

## Share your completed notebook

When you finish:

1. Confirm that all six exercise checks and the complete solver check pass.
2. Complete the reflection.
3. In Colab, click **Share**.
4. Add **michael.d.moffitt@gmail.com** as a Viewer or Commenter.
5. Send the share invite.

Building the complete core solver is the goal. The stretch challenges are optional.